# Enhanced SE Word Embeddings - Interactive Demo

This notebook demonstrates the implementation and comparison of the original Word2Vec approach with an enhanced transformer-based approach for software engineering word embeddings.

## Setup

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

from data_collection import SampleDataGenerator
from preprocessing import SimplePreprocessor
from models import TransformerEmbeddings, OriginalWord2VecModel, ModelComparator
from visualization import EmbeddingVisualizer

print("Setup complete!")

## 1. Data Collection and Preprocessing

In [ ]:
# Get sample SE data
sample_generator = SampleDataGenerator()
sample_texts = sample_generator.get_sample_se_texts()
se_vocabulary = sample_generator.get_se_vocabulary()

print(f"Sample texts: {len(sample_texts)}")
print(f"SE vocabulary: {len(se_vocabulary)} terms")
print("\nFirst sample text:")
print(sample_texts[0][:200] + "...")

In [ ]:
# Preprocess texts
preprocessor = SimplePreprocessor()
processed_texts = preprocessor.preprocess_texts(sample_texts)

print(f"Processed texts: {len(processed_texts)}")
print("\nFirst processed text:")
print(processed_texts[0][:200] + "...")

## 2. Model Training and Loading

In [ ]:
# Train Word2Vec model (original approach)
print("Training Word2Vec model...")
word2vec_model = OriginalWord2VecModel()
word2vec_model.train(processed_texts, vector_size=50, epochs=10)

print(f"Word2Vec vocabulary: {len(word2vec_model.model.wv.key_to_index)} words")

In [ ]:
# Load transformer model (enhanced approach)
print("Loading transformer model...")
transformer_model = TransformerEmbeddings('distilbert-base-uncased')

print(f"Transformer model: {transformer_model.model_name}")
print(f"Device: {transformer_model.device}")

## 3. Model Comparison

In [ ]:
# Define test word pairs
test_pairs = [
    ('software', 'program'),
    ('bug', 'error'),
    ('class', 'object'),
    ('testing', 'debugging'),
    ('design', 'architecture'),
    ('algorithm', 'method'),
    ('framework', 'library'),
    ('database', 'storage')
]

print(f"Test word pairs: {len(test_pairs)}")
for pair in test_pairs:
    print(f"  • {pair[0]} - {pair[1]}")

In [ ]:
# Compare models
comparator = ModelComparator(transformer_model, word2vec_model)
comparison_results = comparator.compare_similarities(test_pairs)
model_stats = comparator.get_model_statistics()
coverage_data = comparator.evaluate_vocabulary_coverage(se_vocabulary[:20])

print("\nComparison completed!")

## 4. Results Analysis

In [ ]:
# Display comparison results
import pandas as pd

df = pd.DataFrame(comparison_results)
print("Similarity Comparison Results:")
print(df[['word_pair', 'word2vec_similarity', 'transformer_similarity', 'improvement']].round(3))

In [ ]:
# Calculate summary statistics
import numpy as np

word2vec_sims = [r['word2vec_similarity'] for r in comparison_results]
transformer_sims = [r['transformer_similarity'] for r in comparison_results]
improvements = [r['improvement'] for r in comparison_results]

print("Summary Statistics:")
print(f"Average Word2Vec similarity: {np.mean(word2vec_sims):.3f}")
print(f"Average Transformer similarity: {np.mean(transformer_sims):.3f}")
print(f"Average improvement: {np.mean(improvements):+.3f}")
print(f"Max improvement: {np.max(improvements):+.3f}")
print(f"Min improvement: {np.min(improvements):+.3f}")

## 5. Visualizations

In [ ]:
# Create visualizations
visualizer = EmbeddingVisualizer('../results')

# Similarity comparison chart
similarity_fig = visualizer.create_similarity_comparison_chart(comparison_results)
similarity_fig.show()

In [ ]:
# Improvement analysis
improvement_fig = visualizer.create_improvement_analysis(comparison_results)
improvement_fig.show()

In [ ]:
# Performance metrics table
metrics_fig = visualizer.create_performance_metrics_table(comparison_results, model_stats)
metrics_fig.show()

In [ ]:
# Vocabulary coverage
coverage_fig = visualizer.create_vocabulary_coverage_chart(coverage_data)
coverage_fig.show()

## 6. Word Similarity Examples

In [ ]:
# Show Word2Vec similar words
test_words = ['software', 'programming', 'testing', 'design']

print("Word2Vec - Most Similar Words:")
for word in test_words:
    similar_words = word2vec_model.get_similar_words(word, topn=3)
    if similar_words:
        print(f"\n{word}:")
        for sim_word, score in similar_words:
            print(f"  • {sim_word}: {score:.3f}")
    else:
        print(f"\n{word}: Not in vocabulary")

## 7. Save Results

In [ ]:
# Save all results
results_data = {
    'comparison_results': comparison_results,
    'model_statistics': model_stats,
    'vocabulary_coverage': coverage_data,
    'summary_statistics': {
        'avg_word2vec_similarity': float(np.mean(word2vec_sims)),
        'avg_transformer_similarity': float(np.mean(transformer_sims)),
        'avg_improvement': float(np.mean(improvements)),
        'max_improvement': float(np.max(improvements)),
        'min_improvement': float(np.min(improvements))
    }
}

os.makedirs('../results', exist_ok=True)
with open('../results/notebook_results.json', 'w') as f:
    json.dump(results_data, f, indent=2)

print("Results saved to ../results/notebook_results.json")
print("\nGenerated visualization files:")
print("• ../results/similarity_comparison.html")
print("• ../results/improvement_analysis.html")
print("• ../results/metrics_table.html")
print("• ../results/vocabulary_coverage.html")

## Conclusion

This notebook demonstrates the implementation and comparison of Word2Vec vs Transformer-based embeddings for software engineering terminology. Key findings:

1. **Performance**: Transformer models show significantly higher semantic similarity scores
2. **Vocabulary**: Better coverage of technical terms through subword tokenization
3. **Context**: Contextual embeddings handle polysemous words more effectively
4. **Trade-offs**: Higher computational requirements and model size

The enhanced approach provides substantial improvements in semantic understanding while requiring more computational resources.